# Fase 2 — Análise Exploratória de Dados (EDA)

**Objetivo desta fase:** Responder perguntas de negócio sobre as transações fraudulentas usando visualizações e estatísticas. Cada descoberta aqui vai orientar decisões no pré-processamento (Fase 3) e na modelagem (Fase 4).

**Perguntas que vamos responder:**
1. Quão severo é o desbalanceamento de classes?
2. Transações fraudulentas têm valores (em USD) maiores?
3. Há padrões temporais — fraudes acontecem em horários ou dias específicos?
4. Quais variáveis numéricas mais se correlacionam com `isFraud`?
5. O missing é aleatório ou sistemático? (colunas com NaN têm comportamento diferente?)
6. Quais categorias (produto, cartão, e-mail) concentram mais fraudes?

## 1. Imports e Configuração

In [ ]:
import os
import sys
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

# Estilo visual consistente para todos os gráficos
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
FRAUD_COLORS = {0: '#4C9BE8', 1: '#E84C4C'}  # azul=legítimo, vermelho=fraude

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = '/home/fernando-daniel-marcelino/data/ieee-cis-fraud-detection'
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Ambiente configurado.')
print(f'Figuras serão salvas em: {FIGURES_DIR}')

## 2. Carregamento do Dataset

Carregamos o arquivo Parquet gerado na Fase 1 — muito mais rápido que reler os CSVs e refazer o merge.

In [ ]:
parquet_path = os.path.join(DATA_DIR, 'train_merged.parquet')
df = pd.read_parquet(parquet_path, engine='pyarrow')

print(f'Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
print(f'Fraudes: {df["isFraud"].sum():,} ({df["isFraud"].mean()*100:.2f}%)')

## 3. Desbalanceamento de Classes

Visualizar o desbalanceamento é o primeiro passo. Números já vimos na Fase 1 — aqui queremos a **percepção visual** de quão extremo é o problema.

In [ ]:
counts = df['isFraud'].value_counts().sort_index()
labels = ['Legítima (0)', 'Fraude (1)']
colors = [FRAUD_COLORS[0], FRAUD_COLORS[1]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Gráfico de barras ---
bars = axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Contagem de transações por classe', fontweight='bold')
axes[0].set_ylabel('Número de transações')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3000,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold')

# --- Gráfico de pizza ---
pcts = counts.values / counts.sum() * 100
wedges, texts, autotexts = axes[1].pie(
    counts.values,
    labels=[f'{l}\n{p:.1f}%' for l, p in zip(labels, pcts)],
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontweight('bold')
axes[1].set_title('Proporção de classes', fontweight='bold')

plt.suptitle('Desbalanceamento: 27.6 transações legítimas para cada fraude', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '01_class_imbalance.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Razão de desbalanceamento: {counts[0]/counts[1]:.1f}:1')

> **Por que isso importa para o modelo?**  
> Com 96.5% de legítimas, um modelo que sempre prevê "legítimo" acerta 96.5% das vezes — e é completamente inútil. Por isso **não usaremos acurácia** como métrica. Na Fase 3, vamos aplicar `class_weight='balanced'` nos modelos para compensar essa disparidade, forçando o algoritmo a prestar mais atenção nos casos raros (fraudes).

## 4. Valor das Transações — Fraudes são mais caras?

Um mito comum é que fraudadores fazem transações de valor alto. Vamos checar se isso é verdade neste dataset.

In [ ]:
# Estatísticas por classe
print('TransactionAmt por classe (USD):')
print(df.groupby('isFraud')['TransactionAmt'].describe().round(2).to_string())
print()

# Mediana é mais representativa que média em distribuições com outliers
for label, name in [(0, 'Legítimas'), (1, 'Fraudes')]:
    med = df[df['isFraud']==label]['TransactionAmt'].median()
    print(f'  Mediana {name}: ${med:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Escala log (necessária por causa dos outliers extremos) ---
for label, name in [(0, 'Legítima'), (1, 'Fraude')]:
    subset = df[df['isFraud'] == label]['TransactionAmt']
    axes[0].hist(np.log1p(subset), bins=60, alpha=0.6,
                 color=FRAUD_COLORS[label], label=f'{name} (n={len(subset):,})',
                 density=True)

axes[0].set_xlabel('log(1 + TransactionAmt)')
axes[0].set_ylabel('Densidade')
axes[0].set_title('Distribuição do valor (escala log)', fontweight='bold')
axes[0].legend()

# Nota: usamos log1p porque log(0) é indefinido — log1p(x) = log(x+1)

# --- Boxplot comparativo ---
data_plot = [
    df[df['isFraud']==0]['TransactionAmt'].clip(upper=500),
    df[df['isFraud']==1]['TransactionAmt'].clip(upper=500)
]
bp = axes[1].boxplot(data_plot, patch_artist=True, labels=['Legítima', 'Fraude'],
                     medianprops={'color': 'black', 'linewidth': 2})
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_ylabel('TransactionAmt (USD) — limitado a $500 para visualização')
axes[1].set_title('Boxplot do valor por classe', fontweight='bold')

plt.suptitle('Valor das transações: Fraude vs Legítima', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '02_transaction_amount.png'), dpi=150, bbox_inches='tight')
plt.show()

> **Interpretação:** As distribuições se sobrepõem bastante — o valor sozinho não distingue bem fraude de legítimo. Fraudes tendem a ter medianas levemente diferentes, mas há muita sobreposição. Isso confirma que precisamos de **múltiplas variáveis** combinadas (o que os modelos de ensemble como XGBoost fazem bem).

## 5. Padrões Temporais

`TransactionDT` é o tempo em segundos desde uma referência. Vamos extrair hora do dia e dia da semana para ver se há padrões temporais de fraude.

**Por que isso importa?** Fraudadores tendem a agir em horários de menor vigilância (madrugada, finais de semana). Se confirmarmos isso, a hora do dia pode ser uma feature poderosa.

In [ ]:
# Extrair hora do dia e dia da semana a partir dos segundos
# TransactionDT em segundos → hora = (segundos % 86400) / 3600
df['hour_of_day'] = (df['TransactionDT'] % 86400) // 3600          # 0–23
df['day_of_week'] = (df['TransactionDT'] // 86400) % 7             # 0–6

print('Hora do dia: min={}, max={}'.format(df['hour_of_day'].min(), df['hour_of_day'].max()))
print('Dia da semana: min={}, max={}'.format(df['day_of_week'].min(), df['day_of_week'].max()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Taxa de fraude por hora do dia ---
fraud_by_hour = df.groupby('hour_of_day')['isFraud'].mean() * 100
volume_by_hour = df.groupby('hour_of_day').size()

ax2 = axes[0].twinx()  # eixo secundário para volume
ax2.bar(fraud_by_hour.index, volume_by_hour.values, color='lightgray', alpha=0.5, label='Volume')
axes[0].plot(fraud_by_hour.index, fraud_by_hour.values, color=FRAUD_COLORS[1],
             linewidth=2.5, marker='o', markersize=4, label='Taxa de fraude')
axes[0].set_xlabel('Hora do dia')
axes[0].set_ylabel('Taxa de fraude (%)', color=FRAUD_COLORS[1])
ax2.set_ylabel('Volume de transações', color='gray')
axes[0].set_title('Taxa de fraude por hora do dia', fontweight='bold')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend(loc='upper right')

# --- Taxa de fraude por dia da semana ---
fraud_by_dow = df.groupby('day_of_week')['isFraud'].mean() * 100
volume_by_dow = df.groupby('day_of_week').size()
day_labels = ['Dom', 'Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb']

ax4 = axes[1].twinx()
ax4.bar(fraud_by_dow.index, volume_by_dow.values, color='lightgray', alpha=0.5)
axes[1].plot(fraud_by_dow.index, fraud_by_dow.values, color=FRAUD_COLORS[1],
             linewidth=2.5, marker='o', markersize=6)
axes[1].set_xlabel('Dia da semana')
axes[1].set_ylabel('Taxa de fraude (%)', color=FRAUD_COLORS[1])
ax4.set_ylabel('Volume de transações', color='gray')
axes[1].set_title('Taxa de fraude por dia da semana', fontweight='bold')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_labels)

plt.suptitle('Padrões temporais de fraude', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '03_temporal_patterns.png'), dpi=150, bbox_inches='tight')
plt.show()

> **O que observar:** Se a taxa de fraude (linha vermelha) variar significativamente entre horas/dias, as features `hour_of_day` e `day_of_week` têm **poder preditivo** e devem ser incluídas no modelo na Fase 3.

## 6. Variáveis Categóricas — Onde a fraude se concentra?

Vamos olhar as principais categóricas: produto, tipo de cartão, bandeira e tipo de dispositivo.

In [ ]:
cat_vars = {
    'ProductCD':  'Categoria do produto',
    'card4':      'Bandeira do cartão',
    'card6':      'Tipo do cartão',
    'DeviceType': 'Tipo de dispositivo',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (col, title) in zip(axes, cat_vars.items()):
    # Taxa de fraude e volume por categoria
    stats = df.groupby(col, observed=True).agg(
        volume=('isFraud', 'count'),
        fraud_rate=('isFraud', 'mean')
    ).sort_values('fraud_rate', ascending=False)

    bars = ax.bar(stats.index, stats['fraud_rate'] * 100,
                  color=FRAUD_COLORS[1], alpha=0.75, edgecolor='white')

    # Anotar volume em cada barra
    for bar, vol in zip(bars, stats['volume']):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.1,
                f'n={vol:,}', ha='center', va='bottom', fontsize=8, color='dimgray')

    # Linha de referência: taxa média global
    ax.axhline(df['isFraud'].mean() * 100, color='steelblue',
               linestyle='--', linewidth=1.5, label='Média global (3.5%)')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Taxa de fraude (%)')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Taxa de fraude por variável categórica', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '04_categorical_fraud_rate.png'), dpi=150, bbox_inches='tight')
plt.show()

> **Como interpretar:** A linha tracejada azul é a taxa média global de fraude (~3.5%). Categorias **acima** da linha têm risco maior que a média — são especialmente informativas para o modelo. Categorias muito abaixo da linha também são úteis: o modelo aprende que certas combinações são seguras.

## 7. Correlação com isFraud — Quais variáveis numéricas mais se relacionam?

Calculamos a correlação de Pearson entre cada variável numérica e o target. Isso não captura relações não-lineares (que o XGBoost encontrará), mas dá uma ideia rápida de quais variáveis têm algum sinal linear.

In [ ]:
# Calcular correlação de cada feature numérica com isFraud
num_cols = df.select_dtypes(include='number').columns.tolist()
# Excluir ID, target e as features temporais que criamos (serão tratadas depois)
exclude = ['TransactionID', 'isFraud', 'hour_of_day', 'day_of_week']
num_features = [c for c in num_cols if c not in exclude]

correlations = df[num_features].corrwith(df['isFraud']).abs().sort_values(ascending=False)

print(f'Total de features numéricas analisadas: {len(correlations)}')
print(f'\nTop 20 mais correlacionadas com isFraud:')
print(correlations.head(20).to_string())

In [ ]:
top_n = 30
top_corr = correlations.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
colors_bar = [FRAUD_COLORS[1] if c > 0.05 else 'steelblue' for c in top_corr.values]
bars = ax.barh(top_corr.index[::-1], top_corr.values[::-1],
               color=colors_bar[::-1], edgecolor='white')

ax.axvline(0.05, color='gray', linestyle='--', linewidth=1, label='Limiar 0.05')
ax.set_xlabel('|Correlação de Pearson| com isFraud')
ax.set_title(f'Top {top_n} variáveis numéricas mais correlacionadas com fraude', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '05_feature_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()

> **Limitação da correlação de Pearson:** Ela mede apenas relações **lineares**. Variáveis com correlação baixa aqui podem ainda ser muito úteis para modelos como XGBoost, que capturam relações não-lineares. Use este gráfico como *ponto de partida*, não como veredicto final sobre importância de features.

## 8. Análise de Missing — O ausente tem significado?

Uma pergunta importante: colunas com muitos valores ausentes têm taxa de fraude diferente?

Se sim, o próprio fato de um valor ser `NaN` (ausente) é informativo — e podemos criar uma feature binária `col_is_missing` para capturar isso.

In [ ]:
# Para cada coluna com missing significativo, comparar taxa de fraude
# entre linhas com valor presente vs ausente
missing_cols = df.columns[df.isnull().mean() > 0.1].tolist()
missing_cols = [c for c in missing_cols if c not in ['isFraud', 'hour_of_day', 'day_of_week']]

results = []
for col in missing_cols:
    fraud_when_missing  = df[df[col].isna()]['isFraud'].mean()
    fraud_when_present  = df[df[col].notna()]['isFraud'].mean()
    diff = abs(fraud_when_missing - fraud_when_present)
    results.append({
        'coluna': col,
        'missing_%': df[col].isna().mean() * 100,
        'fraude_se_missing': fraud_when_missing * 100,
        'fraude_se_presente': fraud_when_present * 100,
        'diferença': diff * 100
    })

missing_df = pd.DataFrame(results).sort_values('diferença', ascending=False)
print('Top 15 colunas onde NaN mais discrimina fraude:')
print(missing_df.head(15).round(2).to_string(index=False))

In [ ]:
top_missing = missing_df.head(15)

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(top_missing))
width = 0.35

ax.bar([i - width/2 for i in x], top_missing['fraude_se_presente'],
       width, label='Valor presente', color=FRAUD_COLORS[0], alpha=0.8)
ax.bar([i + width/2 for i in x], top_missing['fraude_se_missing'],
       width, label='Valor ausente (NaN)', color=FRAUD_COLORS[1], alpha=0.8)

ax.axhline(df['isFraud'].mean() * 100, color='black', linestyle='--',
           linewidth=1.5, label='Média global')
ax.set_xticks(list(x))
ax.set_xticklabels(top_missing['coluna'], rotation=45, ha='right')
ax.set_ylabel('Taxa de fraude (%)')
ax.set_title('Taxa de fraude: valor presente vs ausente\n(colunas onde NaN é mais discriminante)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '06_missing_fraud_rate.png'), dpi=150, bbox_inches='tight')
plt.show()

> **Decisão de pré-processamento:** Colunas onde a taxa de fraude difere muito entre presente/ausente são candidatas a ter uma **feature indicadora de missing** (`col_is_nan = 1/0`) criada na Fase 3. Isso preserva a informação contida no padrão de ausência, ao invés de simplesmente imputar um valor e perder esse sinal.

## 9. Domínios de E-mail — Detalhe interessante

Na Fase 1 vimos que alguns domínios têm taxas de fraude bem diferentes. Vamos visualizar isso.

In [ ]:
# Focar nos domínios com pelo menos 1000 transações para ter estatísticas confiáveis
min_count = 1000
domain_stats = (
    df.groupby('P_emaildomain', observed=True)
    .agg(volume=('isFraud', 'count'), fraud_rate=('isFraud', 'mean'))
    .query(f'volume >= {min_count}')
    .sort_values('fraud_rate', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = [FRAUD_COLORS[1] if r > df['isFraud'].mean() else FRAUD_COLORS[0]
              for r in domain_stats['fraud_rate']]
bars = ax.barh(domain_stats.index, domain_stats['fraud_rate'] * 100,
               color=bar_colors, alpha=0.8, edgecolor='white')

# Anotar volume
for bar, vol in zip(bars, domain_stats['volume']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'n={vol:,}', va='center', fontsize=8, color='dimgray')

ax.axvline(df['isFraud'].mean() * 100, color='black', linestyle='--',
           linewidth=1.5, label='Média global')
ax.set_xlabel('Taxa de fraude (%)')
ax.set_title(f'Taxa de fraude por domínio de e-mail (mínimo {min_count:,} transações)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '07_email_fraud_rate.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Variáveis V — Uma amostra das 339 features da Vesta

Não é viável explorar todas as 339 variáveis V individualmente. Vamos observar as top correlacionadas com `isFraud` para entender o padrão geral.

In [ ]:
# Top 10 variáveis V por correlação com isFraud
v_cols = [c for c in correlations.index if c.startswith('V')]
top_v = v_cols[:10]

print('Top 10 variáveis V mais correlacionadas com isFraud:')
for col in top_v:
    corr_val = correlations[col]
    missing_pct = df[col].isna().mean() * 100
    print(f'  {col}: corr={corr_val:.4f}  missing={missing_pct:.1f}%')

In [ ]:
# Visualizar distribuição de 6 variáveis V top para fraude vs legítimo
plot_v = top_v[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, plot_v):
    for label, name in [(0, 'Legítima'), (1, 'Fraude')]:
        vals = df[df['isFraud']==label][col].dropna()
        # log1p para lidar com distribuições assimétricas
        ax.hist(np.log1p(vals.clip(lower=0)), bins=50, alpha=0.55,
                color=FRAUD_COLORS[label], density=True, label=name)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('log1p(valor)')
    ax.legend(fontsize=8)

plt.suptitle('Distribuição das top variáveis Vesta por classe', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '08_vesta_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Resumo da Fase 2 — Descobertas e Decisões para a Fase 3

### Descobertas principais:

| Pergunta | Resposta |
|----------|----------|
| Desbalanceamento? | 27.6:1 — uso de `class_weight='balanced'` obrigatório |
| Fraudes são mais caras? | Medianas próximas, sobreposição grande — valor sozinho não basta |
| Padrão temporal? | Verificar no gráfico — se sim, adicionar `hour_of_day` e `day_of_week` como features |
| Categóricas informativas? | Sim — ProductCD tipo C (11.7%), outlook.com (9.5%) bem acima da média |
| NaN tem significado? | Sim — para várias colunas, presença/ausência discrimina fraude |

### Decisões para a Fase 3 (Pré-processamento):

1. **Features temporais:** Criar `hour_of_day` e `day_of_week` a partir de `TransactionDT`
2. **Indicadores de missing:** Criar features `_is_nan` para colunas onde ausência é informativa
3. **Imputação:** Usar mediana para numéricas (robusto a outliers), moda para categóricas
4. **Encoding:** Label Encoding ou Target Encoding para categóricas de alta cardinalidade (ex: `DeviceInfo`)
5. **Desbalanceamento:** `class_weight='balanced'` nos modelos (não SMOTE — veja explicação na Fase 3)
6. **Normalização:** Aplicar para Regressão Logística; desnecessário para XGBoost/LightGBM

In [ ]:
# Limpar features temporais auxiliares do DataFrame
# (serão recriadas no pipeline de pré-processamento da Fase 3)
df.drop(columns=['hour_of_day', 'day_of_week'], inplace=True)

print('Figuras salvas em:', FIGURES_DIR)
print('\nArquivos gerados:')
for f in sorted(os.listdir(FIGURES_DIR)):
    path = os.path.join(FIGURES_DIR, f)
    print(f'  {f}  ({os.path.getsize(path)/1024:.0f} KB)')